In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

bronze_game = spark.table("workspace.bronze_nba.game")

print(f"Bronze rows: {bronze_game.count():,}")
print(f"Bronze columns: {len(bronze_game.columns)}")
bronze_game.printSchema()

Bronze rows: 65,698
Bronze columns: 57
root
 |-- season_id: string (nullable = true)
 |-- team_id_home: string (nullable = true)
 |-- team_abbreviation_home: string (nullable = true)
 |-- team_name_home: string (nullable = true)
 |-- game_id: string (nullable = true)
 |-- game_date: string (nullable = true)
 |-- matchup_home: string (nullable = true)
 |-- wl_home: string (nullable = true)
 |-- min: string (nullable = true)
 |-- fgm_home: string (nullable = true)
 |-- fga_home: string (nullable = true)
 |-- fg_pct_home: string (nullable = true)
 |-- fg3m_home: string (nullable = true)
 |-- fg3a_home: string (nullable = true)
 |-- fg3_pct_home: string (nullable = true)
 |-- ftm_home: string (nullable = true)
 |-- fta_home: string (nullable = true)
 |-- ft_pct_home: string (nullable = true)
 |-- oreb_home: string (nullable = true)
 |-- dreb_home: string (nullable = true)
 |-- reb_home: string (nullable = true)
 |-- ast_home: string (nullable = true)
 |-- stl_home: string (nullable = true)

In [0]:
# Stat columns that exist for both home and away sides
stat_cols = [
    "team_id", "team_abbreviation", "team_name", "matchup", "wl",
    "fgm", "fga", "fg_pct", "fg3m", "fg3a", "fg3_pct",
    "ftm", "fta", "ft_pct", "oreb", "dreb", "reb",
    "ast", "stl", "blk", "tov", "pf", "pts", "plus_minus"
]

# Shared columns that appear once per game, not per team
game_cols = ["game_id", "game_date", "season_id", "season_type", "min"]

def select_side(df, side):
    cols = [F.col(c) for c in game_cols]
    cols += [F.col(f"{c}_{side}").alias(c) for c in stat_cols]
    cols += [F.lit(side).alias("home_away")]
    return df.select(cols)

home = select_side(bronze_game, "home")
away = select_side(bronze_game, "away")

# Stack both sides 
silver_game = home.unionByName(away)

silver_game = silver_game \
    .withColumn("game_date", F.to_date("game_date")) \
    .withColumn("season_id", F.col("season_id").cast("integer")) \
    .withColumn("fgm", F.col("fgm").cast("double")) \
    .withColumn("fga", F.col("fga").cast("double")) \
    .withColumn("fg_pct", F.col("fg_pct").cast("double")) \
    .withColumn("fg3m", F.col("fg3m").cast("double")) \
    .withColumn("fg3a", F.col("fg3a").cast("double")) \
    .withColumn("fg3_pct", F.col("fg3_pct").cast("double")) \
    .withColumn("ftm", F.col("ftm").cast("double")) \
    .withColumn("fta", F.col("fta").cast("double")) \
    .withColumn("ft_pct", F.col("ft_pct").cast("double")) \
    .withColumn("oreb", F.col("oreb").cast("double")) \
    .withColumn("dreb", F.col("dreb").cast("double")) \
    .withColumn("reb", F.col("reb").cast("double")) \
    .withColumn("ast", F.col("ast").cast("double")) \
    .withColumn("stl", F.col("stl").cast("double")) \
    .withColumn("blk", F.col("blk").cast("double")) \
    .withColumn("tov", F.col("tov").cast("double")) \
    .withColumn("pf", F.col("pf").cast("double")) \
    .withColumn("pts", F.col("pts").cast("double")) \
    .withColumn("plus_minus", F.col("plus_minus").cast("integer"))


silver_game = silver_game.withColumn(
    "season_year",
    F.col("season_id") % 10000
)

print(f"Silver rows: {silver_game.count():,}")
print(f"Silver columns: {len(silver_game.columns)}")
silver_game.show(3)

Silver rows: 131,396
Silver columns: 31
+----------+----------+---------+-----------+---+----------+-----------------+--------------------+-----------+---+----+----+------+----+----+-------+----+----+------+----+----+----+----+----+---+----+----+----+----------+---------+-----------+
|   game_id| game_date|season_id|season_type|min|   team_id|team_abbreviation|           team_name|    matchup| wl| fgm| fga|fg_pct|fg3m|fg3a|fg3_pct| ftm| fta|ft_pct|oreb|dreb| reb| ast| stl|blk| tov|  pf| pts|plus_minus|home_away|season_year|
+----------+----------+---------+-----------+---+----------+-----------------+--------------------+-----------+---+----+----+------+----+----+-------+----+----+------+----+----+----+----+----+---+----+----+----+----------+---------+-----------+
|0010900057|2009-10-14|    12009| Pre Season|240|1610612750|              MIN|Minnesota Timberw...|MIN vs. CHI|  L|30.0|77.0|  0.39| 2.0|18.0|  0.111|32.0|49.0| 0.653|14.0|32.0|46.0|16.0|14.0|3.0|18.0|22.0|94.0|        -5|   

In [0]:
silver_game.groupBy("season_type").count().orderBy("count", ascending=False).show()

+--------------+------+
|   season_type| count|
+--------------+------+
|Regular Season|120384|
|      Playoffs|  7684|
|    Pre Season|  3072|
|      All Star|   130|
|      All-Star|   126|
+--------------+------+



In [0]:
#only Regular Season and Playoffs 
silver_game_clean = silver_game.filter(
    F.col("season_type").isin(["Regular Season", "Playoffs"])
)

print(f"Rows after filtering: {silver_game_clean.count():,}")

# Write to Silver -- partitioned by season_type for query efficiency
silver_game_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver_nba.game")

print("Written to workspace.silver_nba.game")


spark.table("workspace.silver_nba.game") \
    .groupBy("season_type") \
    .count() \
    .show()

Rows after filtering: 128,068
Written to workspace.silver_nba.game
+--------------+------+
|   season_type| count|
+--------------+------+
|Regular Season|120384|
|      Playoffs|  7684|
+--------------+------+



In [0]:
spark.sql("""
    OPTIMIZE workspace.silver_nba.game 
    ZORDER BY (season_year, team_id)
""")

print("Z-ordering applied on season_year and team_id")

Z-ordering applied on season_year and team_id


In [0]:
spark.table("workspace.silver_nba.game") \
    .filter(
        (F.col("team_abbreviation") == "GSW") & 
        (F.col("season_year") == 2015) & 
        (F.col("season_type") == "Regular Season")
    ) \
    .agg(
        F.count("game_id").alias("games"),
        F.avg("pts").alias("avg_pts"),
        F.avg("fg3a").alias("avg_3pt_attempts"),
        F.sum(F.when(F.col("wl") == "W", 1).otherwise(0)).alias("wins")
    ) \
    .show()

+-----+------------------+------------------+----+
|games|           avg_pts|  avg_3pt_attempts|wins|
+-----+------------------+------------------+----+
|   82|114.89024390243902|31.609756097560975|  73|
+-----+------------------+------------------+----+



In [0]:
bronze_other = spark.table("workspace.bronze_nba.other_stats")
bronze_other.show(2)
bronze_other.printSchema()

+----------+---------+------------+----------------------+--------------+--------------+-------------------+-----------+-----------------+------------+----------+-------------------+--------------------+------------------+---------------+------------+----------------------+--------------+--------------+-------------------+-----------+-----------------+-------------------+--------------------+------------------+---------------+--------------------+----------+
|   game_id|league_id|team_id_home|team_abbreviation_home|team_city_home|pts_paint_home|pts_2nd_chance_home|pts_fb_home|largest_lead_home|lead_changes|times_tied|team_turnovers_home|total_turnovers_home|team_rebounds_home|pts_off_to_home|team_id_away|team_abbreviation_away|team_city_away|pts_paint_away|pts_2nd_chance_away|pts_fb_away|largest_lead_away|team_turnovers_away|total_turnovers_away|team_rebounds_away|pts_off_to_away|        _ingested_at|   _source|
+----------+---------+------------+----------------------+--------------+-

In [0]:
silver_other = home.unionByName(away)

silver_other = silver_other \
    .withColumn("pts_paint", F.expr("try_cast(pts_paint as integer)")) \
    .withColumn("pts_2nd_chance", F.expr("try_cast(pts_2nd_chance as integer)")) \
    .withColumn("pts_fb", F.expr("try_cast(pts_fb as integer)")) \
    .withColumn("largest_lead", F.expr("try_cast(largest_lead as integer)")) \
    .withColumn("lead_changes", F.expr("try_cast(lead_changes as integer)")) \
    .withColumn("times_tied", F.expr("try_cast(times_tied as integer)")) \
    .withColumn("team_turnovers", F.expr("try_cast(team_turnovers as double)")) \
    .withColumn("total_turnovers", F.expr("try_cast(total_turnovers as double)")) \
    .withColumn("team_rebounds", F.expr("try_cast(team_rebounds as double)")) \
    .withColumn("pts_off_to", F.expr("try_cast(pts_off_to as integer)"))

silver_other.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver_nba.other_stats")

print(f"Rows written: {silver_other.count():,}")
spark.sql("OPTIMIZE workspace.silver_nba.other_stats ZORDER BY (team_id)")
print("Done")

Rows written: 56,542
Done


In [0]:
from pyspark.sql import functions as F

silver_game = spark.table("workspace.silver_nba.game")

pace_cols = ["fga", "oreb", "tov", "fta", "fg3a", "pts"]
total = silver_game.filter(F.col("season_type") == "Regular Season").count()

print(f"Total regular season team-games: {total:,}\n")
print(f"{'column':15s} {'nulls':>8s} {'null_pct':>10s}")
print("-" * 36)

for col in pace_cols:
    nulls = silver_game.filter(
        (F.col("season_type") == "Regular Season") & 
        F.col(col).isNull()
    ).count()
    print(f"{col:15s} {nulls:>8,} {nulls/total*100:>9.1f}%")

Total regular season team-games: 120,384

column             nulls   null_pct
------------------------------------
fga                    0       0.0%
oreb                   0       0.0%
tov                    0       0.0%
fta                    0       0.0%
fg3a                   0       0.0%
pts                    0       0.0%


In [0]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()

silver_game = spark.table("workspace.silver_nba.game") \
    .filter(F.col("season_type") == "Regular Season")

# Calculate pace and 3pt rate at game level first, then aggregate to season
game_with_pace = silver_game.withColumn(
    "possessions",
    F.col("fga") - F.col("oreb") + F.col("tov") + (0.44 * F.col("fta"))
).withColumn(
    "fg3_rate",
    F.expr("try_divide(fg3a, fga)")  # returns NULL if fga = 0
)

# Aggregate to team-season level
gold_pace = game_with_pace.groupBy("team_id", "team_abbreviation", "season_year") \
    .agg(
        F.count("game_id").alias("games_played"),
        F.avg("possessions").alias("pace"),           # avg possessions per game
        F.avg("fg3_rate").alias("three_pt_rate"),     # avg 3pt attempt rate
        F.avg("fg3a").alias("avg_fg3a"),              # raw 3pt attempts per game
        F.avg("pts").alias("avg_pts"),                # points per game
        F.avg("tov").alias("avg_tov"),                # turnovers per game
        F.sum(F.when(F.col("wl") == "W", 1).otherwise(0)).alias("wins"),
        F.avg("fta").alias("avg_fta"),                # free throw attempts per game
        F.avg("oreb").alias("avg_oreb")               # offensive rebounds per game
    ) \
    .filter(F.col("games_played") >= 20)  # filter partial seasons / lockout years with < 20 games

gold_pace = gold_pace.withColumn(
    "win_pct", F.col("wins") / F.col("games_played")
)

gold_pace.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold_nba.pace_revolution")

spark.sql("OPTIMIZE workspace.gold_nba.pace_revolution ZORDER BY (season_year, team_id)")

print(f"Rows written: {gold_pace.count():,}")
gold_pace.orderBy("season_year", "team_abbreviation").show(5)

Rows written: 1,519
+----------+-----------------+-----------+------------+----+-------------+--------+------------------+-------+----+-------+--------+-------------------+
|   team_id|team_abbreviation|season_year|games_played|pace|three_pt_rate|avg_fg3a|           avg_pts|avg_tov|wins|avg_fta|avg_oreb|            win_pct|
+----------+-----------------+-----------+------------+----+-------------+--------+------------------+-------+----+-------+--------+-------------------+
|1610610034|              BOM|       1946|          61| NaN|          NaN|     NaN| 66.62295081967213|    NaN|  38|    NaN|     NaN| 0.6229508196721312|
|1610612738|              BOS|       1946|          60| NaN|          NaN|     NaN|60.083333333333336|    NaN|  22|    NaN|     NaN|0.36666666666666664|
|1610610025|              CHS|       1946|          61| NaN|          NaN|     NaN|              77.0|    NaN|  39|    NaN|     NaN|  0.639344262295082|
|1610610026|              CLR|       1946|          60| NaN|  

In [0]:
spark.table("workspace.gold_nba.pace_revolution") \
    .filter(F.col("three_pt_rate").isNotNull()) \
    .groupBy("season_year") \
    .agg(
        F.round(F.avg("pace"), 1).alias("league_avg_pace"),
        F.round(F.avg("three_pt_rate"), 3).alias("league_avg_3pt_rate"),
        F.round(F.avg("avg_fg3a"), 1).alias("league_avg_fg3a")
    ) \
    .orderBy("season_year") \
    .show(50)

+-----------+---------------+-------------------+---------------+
|season_year|league_avg_pace|league_avg_3pt_rate|league_avg_fg3a|
+-----------+---------------+-------------------+---------------+
|       1946|            NaN|                NaN|            NaN|
|       1947|            NaN|                NaN|            NaN|
|       1948|            NaN|                NaN|            NaN|
|       1949|            NaN|                NaN|            NaN|
|       1950|            NaN|                NaN|            NaN|
|       1951|            NaN|                NaN|            NaN|
|       1952|            NaN|                NaN|            NaN|
|       1953|            NaN|                NaN|            NaN|
|       1954|            NaN|                NaN|            NaN|
|       1955|            NaN|                NaN|            NaN|
|       1956|            NaN|                NaN|            NaN|
|       1957|            NaN|                NaN|            NaN|
|       19

In [0]:
spark.table("workspace.gold_nba.pace_revolution") \
    .filter(F.col("three_pt_rate").isNotNull()) \
    .groupBy("season_year") \
    .agg(
        F.round(F.avg("pace"), 1).alias("league_avg_pace"),
        F.round(F.avg("three_pt_rate"), 3).alias("league_avg_3pt_rate"),
        F.round(F.avg("avg_fg3a"), 1).alias("league_avg_fg3a")
    ) \
    .filter(F.col("season_year") >= 2002) \
    .orderBy("season_year") \
    .show(50)

+-----------+---------------+-------------------+---------------+
|season_year|league_avg_pace|league_avg_3pt_rate|league_avg_fg3a|
+-----------+---------------+-------------------+---------------+
|       2002|           94.4|              0.182|           14.7|
|       2003|           93.3|              0.187|           14.9|
|       2004|           94.3|              0.196|           15.8|
|       2005|           93.8|              0.202|           16.0|
|       2006|           95.2|              0.212|           16.9|
|       2007|           95.4|              0.222|           18.1|
|       2008|           94.8|              0.224|           18.1|
|       2009|           95.8|              0.222|           18.1|
|       2010|           95.3|              0.222|           18.0|
|       2011|           94.5|              0.226|           18.4|
|       2013|           97.1|               0.26|           21.5|
|       2014|           97.1|              0.268|           22.4|
|       20